# SecRAG-Gate DeBERTa Training

This Colab notebook trains `microsoft/deberta-v3-base` as a factual versus injection classifier using three sources:

1. `deepset/prompt-injections` for direct injections.
2. BIPIA for indirect injections in documents.
3. The uploaded `poisoned_chunks.csv` from the SecRAG project.

Manual input is limited to uploading `poisoned_chunks.csv`. The notebook downloads the other datasets automatically, evaluates on separate deepset/BIPIA test data, saves the model, and downloads a zip for the local project.

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn sentencepiece protobuf jsonlines huggingface_hub

In [ ]:
import json
import random
import shutil
import subprocess
from pathlib import Path

import jsonlines
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, precision_recall_fscore_support)
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          Trainer, TrainingArguments)

SEED = 42
MODEL_NAME = 'microsoft/deberta-v3-base'
OUTPUT_DIR = Path('/content/secrag-deberta-final')
BIPIA_DIR = Path('/content/BIPIA/benchmark')
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Upload the project dataset

Upload only `poisoned_chunks.csv` when prompted.

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_candidates = [name for name in uploaded if name.lower().endswith('.csv')]
if not csv_candidates:
    raise FileNotFoundError('Upload poisoned_chunks.csv')
CSV_PATH = Path('/content') / csv_candidates[0]
csv_preview = pd.read_csv(CSV_PATH)
required_columns = {'poisoned text', 'label'}
missing = required_columns - set(csv_preview.columns)
if missing:
    raise ValueError(f'CSV is missing columns: {sorted(missing)}')
print('Using:', CSV_PATH)
print('Rows:', len(csv_preview), '| columns:', list(csv_preview.columns))

In [ ]:
if not Path('/content/BIPIA').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/microsoft/BIPIA.git', '/content/BIPIA'], check=True)
print('BIPIA ready:', BIPIA_DIR)
required_bipia_files = [
    BIPIA_DIR / 'email/train.jsonl', BIPIA_DIR / 'email/test.jsonl',
    BIPIA_DIR / 'table/train.jsonl', BIPIA_DIR / 'table/test.jsonl',
    BIPIA_DIR / 'text_attack_train.json', BIPIA_DIR / 'text_attack_test.json',
]
missing_bipia = [str(path) for path in required_bipia_files if not path.exists()]
if missing_bipia:
    raise FileNotFoundError(f'Missing BIPIA files: {missing_bipia}')

## Build leakage-free train and test data

The uploaded project CSV is training augmentation only. Test metrics use deepset and BIPIA rows that are not used for training. Any accidental exact duplicates are removed from the test set.

In [ ]:
def load_contexts(path):
    with jsonlines.open(path) as reader:
        return [row['context'] for row in reader]

def load_attacks(path):
    with path.open(encoding='utf-8') as handle:
        raw = json.load(handle)
    return [attack for variants in raw.values() for attack in variants]

def build_bipia_split(context_files, attack_file, poisoned_count, seed):
    rng = random.Random(seed)
    contexts = [text for path in context_files for text in load_contexts(path)]
    attacks = load_attacks(attack_file)
    insertions = (
        lambda context, attack: f'{context}\n{attack}',
        lambda context, attack: f'{attack}\n{context}',
    )
    poisoned = set()
    while len(poisoned) < poisoned_count:
        poisoned.add(rng.choice(insertions)(rng.choice(contexts), rng.choice(attacks)))
    return list(poisoned), list(dict.fromkeys(contexts))

def build_data():
    direct = load_dataset('deepset/prompt-injections')
    direct_train = direct['train'].to_pandas()[['text', 'label']]
    direct_test = direct['test'].to_pandas()[['text', 'label']]
    direct_train['source'] = 'deepset'
    direct_test['source'] = 'deepset'

    train_poison, train_clean = build_bipia_split(
        [BIPIA_DIR / 'email/train.jsonl', BIPIA_DIR / 'table/train.jsonl'],
        BIPIA_DIR / 'text_attack_train.json', 1500, SEED)
    test_poison, test_clean = build_bipia_split(
        [BIPIA_DIR / 'email/test.jsonl', BIPIA_DIR / 'table/test.jsonl'],
        BIPIA_DIR / 'text_attack_test.json', 300, SEED)
    bipia_train = pd.DataFrame({
        'text': train_clean + train_poison,
        'label': [0] * len(train_clean) + [1] * len(train_poison),
        'source': 'bipia',
    })
    bipia_test = pd.DataFrame({
        'text': test_clean + test_poison,
        'label': [0] * len(test_clean) + [1] * len(test_poison),
        'source': 'bipia',
    })

    project_poison = pd.read_csv(CSV_PATH)[['poisoned text', 'label']].rename(columns={'poisoned text': 'text'})
    project_poison['label'] = 1
    project_poison['source'] = 'project_csv'

    train_df = pd.concat([direct_train, bipia_train, project_poison], ignore_index=True)
    test_df = pd.concat([direct_test, bipia_test], ignore_index=True)
    train_df = train_df.drop_duplicates('text').sample(frac=1, random_state=SEED).reset_index(drop=True)
    test_df = test_df.drop_duplicates('text').sample(frac=1, random_state=SEED).reset_index(drop=True)
    train_texts = set(train_df['text'])
    overlap = test_df['text'].isin(train_texts)
    if overlap.any():
        print('Removing duplicate test rows:', int(overlap.sum()))
        test_df = test_df[~overlap].reset_index(drop=True)
    return train_df, test_df

train_df, test_df = build_data()
print('Train:', train_df.shape, train_df['label'].value_counts().to_dict())
print('Test:', test_df.shape, test_df['label'].value_counts().to_dict())
print('Train sources:', train_df['source'].value_counts().to_dict())
print('Test sources:', test_df['source'].value_counts().to_dict())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=256)

train_dataset = Dataset.from_pandas(train_df[['text', 'label']], preserve_index=False).map(tokenize, batched=True)
test_dataset = Dataset.from_pandas(test_df[['text', 'label']], preserve_index=False).map(tokenize, batched=True)
train_dataset = train_dataset.rename_column('label', 'labels')
test_dataset = test_dataset.rename_column('label', 'labels')
columns = ['input_ids', 'attention_mask', 'labels']
train_dataset.set_format(type='torch', columns=columns)
test_dataset.set_format(type='torch', columns=columns)
print('Tokenization complete')

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'FACTUAL', 1: 'INJECTION'},
    label2id={'FACTUAL': 0, 'INJECTION': 1},
    dtype=torch.float32,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary', zero_division=0)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

training_args = TrainingArguments(
    output_dir='/content/secrag-training-checkpoints',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)
print('Training configured. Starting now...')
trainer.train()

In [ ]:
evaluation = trainer.evaluate()
print('Overall metrics:', evaluation)

predictions = trainer.predict(test_dataset)
predicted_labels = np.argmax(predictions.predictions, axis=-1)
test_report = test_df.reset_index(drop=True).copy()
test_report['pred'] = predicted_labels
print(classification_report(test_report['label'], test_report['pred'], target_names=['FACTUAL', 'INJECTION'], zero_division=0))
print('Overall confusion matrix:')
print(confusion_matrix(test_report['label'], test_report['pred']))
for source in test_report['source'].unique():
    subset = test_report[test_report['source'] == source]
    print(f'\n{source} (n={len(subset)})')
    print(classification_report(subset['label'], subset['pred'], target_names=['FACTUAL', 'INJECTION'], zero_division=0))
    print(confusion_matrix(subset['label'], subset['pred']))

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
summary = {
    'model': MODEL_NAME,
    'seed': SEED,
    'train_rows': len(train_df),
    'test_rows': len(test_df),
    'test_metrics': evaluation,
    'class_mapping': {'0': 'FACTUAL', '1': 'INJECTION'},
}
(OUTPUT_DIR / 'training_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
archive = shutil.make_archive('/content/secrag-deberta-final', 'zip', root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name)
print('Saved:', OUTPUT_DIR)
print('Archive:', archive)

## Download the trained model

Run the next cell. Extract the downloaded zip locally so that `models/secrag-deberta-final/config.json` exists.

In [ ]:
from google.colab import files
files.download('/content/secrag-deberta-final.zip')